# Part 1: Foundations

Covers How Machine Learning Works through Setting Up a Workspace. The [README](README.md) has the same content with the full table of contents; datasets and versions are listed in its Setup section.

In [1]:
# Shared setup: imports and paths used throughout. Run this cell first.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")      # keep TensorFlow's start-up logs quiet
from pathlib import Path
import logging, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="(Degrees of freedom|divide by zero|invalid value)")  # sparse correlations
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)  # Prophet's backend is chatty
logging.getLogger("tensorflow").setLevel(logging.ERROR)
DATA = Path("data")
IMAGES = Path("images")                                   # animations are written here
MODELS = Path("models")                                   # saved models
IMAGES.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)
np.random.seed(42)                                        # for the classic np.random API
rng = np.random.default_rng(42)                           # for new code
plt.rcParams.update({"figure.dpi": 80, "axes.spines.top": False, "axes.spines.right": False})

print("pandas", pd.__version__, "| NumPy", np.__version__)

pandas 3.0.5 | NumPy 2.5.3


<a id="1-how-machine-learning-works"></a>

## 1. How Machine Learning Works

Before any code, it helps to be precise about what machine learning is, how it relates to the other terms that get thrown
around, and which kinds of problems it solves.

<a id="rules-written-by-data"></a>

### Rules Written by Data

In normal programming you write the rules. You give the computer an input and a set of instructions, and it produces an
output. A recipe is a good analogy: ingredients plus steps give you a dish.

Machine learning flips that around. You give the computer inputs *and* the outputs you want, and it works out the
instructions for itself. That's useful wherever writing the rules by hand is impractical: recognizing faces, recommending
the next video, estimating a fair price for a used machine.

```mermaid
flowchart LR
    subgraph Traditional programming
        A[Input] --> P[Rules you write] --> O[Output]
    end
    subgraph Machine learning
        I[Inputs] --> M[Algorithm]
        Y[Known outputs] --> M
        M --> R[Learned rules = model]
    end
```

A trained model is a function learned from data, `f(features) → prediction`. The rest of this post is about making that
function generalize to data it has never seen.

Here is what "learning" looks like in its simplest form. A straight line `y = w·x + b` starts with random parameters, and
gradient descent nudges `w` and `b` a little at a time in whichever direction reduces the error, until the line fits the data.

In [2]:
# Animate gradient descent fitting a line, and save it as a GIF.
from matplotlib.animation import FuncAnimation, PillowWriter

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 60)
y = 3 * x + 4 + rng.normal(0, 3, 60)                 # true relationship: y = 3x + 4, plus noise

w, b, lr, history = 0.0, 0.0, 0.01, []
for _ in range(60):
    error = (w * x + b) - y
    history.append((w, b, np.mean(error ** 2)))
    w -= lr * 2 * np.mean(error * x)                 # gradient of mean squared error w.r.t. w
    b -= lr * 2 * np.mean(error)                     # ... and w.r.t. b

fig, (ax_fit, ax_loss) = plt.subplots(1, 2, figsize=(9, 3.5))
ax_fit.scatter(x, y, s=12, alpha=0.6)
line, = ax_fit.plot([], [], color="crimson", lw=2)
ax_fit.set(xlim=(0, 10), ylim=(y.min() - 5, y.max() + 5), title="Fitting y = w·x + b", xlabel="x", ylabel="y")
ax_loss.set(xlim=(0, 60), ylim=(0, history[0][2] * 1.05), title="Mean squared error", xlabel="step")
loss_line, = ax_loss.plot([], [], color="crimson")
fig.tight_layout()

def update(frame):
    w_, b_, _ = history[frame]
    line.set_data([0, 10], [b_, w_ * 10 + b_])
    loss_line.set_data(range(frame + 1), [h[2] for h in history[: frame + 1]])
    ax_fit.set_title(f"step {frame}: w={w_:.2f}, b={b_:.2f}")
    return line, loss_line

FuncAnimation(fig, update, frames=len(history), interval=80).save(IMAGES / "gradient-descent.gif",
                                                                  writer=PillowWriter(fps=12))
plt.close(fig)
print(f"final: w={history[-1][0]:.2f}, b={history[-1][1]:.2f}  (true values: 3, 4)")

final: w=3.40, b=1.51  (true values: 3, 4)


![Gradient descent fitting a line](images/gradient-descent.gif)

The intercept is still far from 4 after 60 steps because `b` learns slowly at this learning rate. That's a small preview of
why tuning matters.

<a id="ai-machine-learning-deep-learning-and-data-science"></a>

### AI, Machine Learning, Deep Learning and Data Science

These terms overlap, and people often use them loosely:

```mermaid
flowchart TB
    AI["Artificial intelligence: machines acting intelligently"] --> ML["Machine learning: learning patterns from data"]
    ML --> DL["Deep learning: neural networks, especially for images, text, audio"]
    DS["Data science: analysing data to make decisions"] -.overlaps.- ML
```

- **Artificial intelligence** is the broad goal. Everything that exists today is *narrow* AI, meaning very good at one task
  (playing chess, spotting tumours) rather than general human-level intelligence.
- **Machine learning** is one way of achieving it: systems that improve from data instead of being explicitly programmed.
- **Deep learning** is the part of machine learning that uses multi-layer neural networks. It dominates on unstructured
  data.
- **Data science** is the practice of analysing data to answer questions. Sometimes that involves a model, and sometimes a
  well-chosen chart is enough.

The practical difference between classic machine learning and deep learning is who designs the features. A spam filter
built with Naive Bayes needs us to turn text into word counts first. A convolutional network takes raw pixels and learns
edges, textures and shapes on its own, at the cost of needing much more data and compute.

The reason this all took off recently is mostly about data and compute. Businesses went from paper to spreadsheets to
databases to "big data", and cheap storage and GPUs made it practical to let algorithms find the patterns.

<a id="types-of-machine-learning"></a>

### Types of Machine Learning

| Type | You have | The model learns to | Examples |
|---|---|---|---|
| **Supervised** | data **with labels** | map inputs to known outputs | classification (disease yes/no, dog breed), regression (house price) |
| **Unsupervised** | data **without labels** | find structure on its own | clustering customers, "people who bought X also bought Y" |
| **Transfer learning** | a model trained on a related problem | reuse what it learned | adapting an ImageNet model to recognise dog breeds |
| **Reinforcement** | an environment with rewards | choose actions that maximize reward | game-playing agents, robotics |

Within supervised learning, **classification** predicts a category. It can be *binary* (two classes) or *multi-class*
(many classes), and in *multi-label* problems one example can have several labels. **Regression** predicts a number. Most
business problems are one of these two.

Not every problem needs machine learning. If a handful of hand-written rules solves it reliably, write the rules. They are
easier to test and to explain.

<a id="2-a-framework-for-any-machine-learning-project"></a>

## 2. A Framework for Any Machine Learning Project

A repeatable framework turns a vague request ("can we use AI for this?") into concrete steps. The six steps below aren't
a strict sequence. You'll loop back constantly, but they give you a checklist.

<a id="six-steps-from-question-to-model"></a>

### Six Steps from Question to Model

```mermaid
flowchart LR
    P[1. Problem definition] --> D[2. Data] --> E[3. Evaluation] --> F[4. Features] --> M[5. Modelling] --> X[6. Experimentation]
    X -.revisit.-> P
    X -.revisit.-> D
    X -.revisit.-> F
```

1. **Problem definition.** What exactly are we predicting? Is it supervised or unsupervised, classification or regression?
2. **Data.** What do we have? Is it structured (tables) or unstructured (images, text)? Static files or a live stream?
3. **Evaluation.** What does success look like? Pick a metric and a threshold that would make the project worthwhile, for
   example "95% accuracy at detecting heart disease".
4. **Features.** What do we know about the inputs? Which columns could plausibly predict the target?
5. **Modelling.** Which model suits the problem? How will we train, tune and compare it?
6. **Experimentation.** What could we try next? Change one thing, re-evaluate, repeat.

<a id="data-metrics-and-features"></a>

### Data, Metrics and Features

**Data** comes in two broad shapes. *Structured* data has rows and columns of numbers and categories, like a table of
patient records. *Unstructured* data has no fixed schema: images, audio, free text. It can also be *static* (a CSV export)
or *streaming* (prices that update every second). Most projects start static and only move to streaming once a model has
proven itself.

**Metrics** should reflect the real cost of mistakes. Typical choices:

| Problem | Common metrics |
|---|---|
| Classification | accuracy, precision, recall, F1, ROC AUC |
| Regression | mean absolute error (MAE), (root) mean squared error, R² |
| Recommendation | precision at K |

In medical screening, a missed case (a false negative) is far worse than a false alarm, so recall matters more than raw
accuracy. Decide this up front, and settle what "good enough" means, so you know when to stop tinkering.

**Features** are the input columns used to predict the **target**. They can be numerical (weight), categorical (sex,
smoker yes/no) or *derived*, meaning engineered from other columns (for example "visits in the last year"). A feature only
helps if most samples actually have it. A column filled in for 10 patients out of 100 gives the model very little to learn
from. That proportion is called *feature coverage*.

<a id="train-validation-and-test-sets"></a>

### Train, Validation and Test Sets

If there's one concept to internalize, it's this one. A model is only useful if it **generalizes** to data it hasn't seen,
and the only way to measure that is to hold data back.

Think of studying for an exam: you learn from the textbook (the **training set**), check yourself on a practice exam (the
**validation set**), and prove yourself on the final exam (the **test set**). If you've seen the final exam in advance, a
perfect score proves nothing.

```mermaid
flowchart LR
    ALL[All labelled data] --> TR["Training set (70-80%): fit the model"]
    ALL --> VA["Validation set (10-15%): tune hyperparameters, compare models"]
    ALL --> TE["Test set (10-15%): final, one-time evaluation"]
```

```python
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
```

On small datasets, **cross-validation** replaces a fixed validation set. With time-ordered data, split by time and never
randomly. Both come up later.

<a id="choosing-and-tuning-a-model"></a>

### Choosing and Tuning a Model

A reliable rule of thumb: for **structured** data, ensembles of decision trees (random forests, gradient boosting such
as XGBoost, LightGBM, CatBoost) are hard to beat. For **unstructured** data, use deep learning, usually through transfer
learning.

Every model has **parameters** it learns (the weights) and **hyperparameters** you choose (number of trees, tree depth,
learning rate). Tune hyperparameters on the training and validation data, and only touch the test set once, at the very end.

Start small. Train on a subset first so each experiment takes seconds rather than minutes, and scale up once the pipeline
works. The faster your experiment loop, the more ideas you can test.

<a id="underfitting-overfitting-and-the-goldilocks-zone"></a>

### Underfitting, Overfitting and the Goldilocks Zone

Comparing training and test performance tells you which problem you have:

| Symptom | Training score | Test score | What to try |
|---|---|---|---|
| **Underfitting** | low | low | a more complex model, better features, train longer |
| **Good fit** | high | close to training | ship it |
| **Overfitting** | very high | much lower | more data, a simpler model, regularization, early stopping |

The animation below fits polynomials of increasing degree to 20 noisy points from a sine curve. Degree 1 underfits, since
a line can't follow the curve. Around degree 3–5 the fit is right. At high degrees the curve threads every training point,
and the validation error explodes.

The other two classic traps are **data leakage**, where information from the test set seeps into training (for example
by fitting a scaler on all the data before splitting), and **data mismatch**, where the training data doesn't look like
the data the model will face in production. Both make results look better than they are. Leakage gets a full treatment in
the [Data Preparation reference](https://github.com/rokon-uddin/data-preparation-reference).

In [3]:
# Animate underfitting -> good fit -> overfitting with polynomials of increasing degree.
from matplotlib.animation import FuncAnimation, PillowWriter

rng = np.random.default_rng(1)
x_train = np.sort(rng.uniform(0, 1, 20)); y_train = np.sin(2 * np.pi * x_train) + rng.normal(0, 0.2, 20)
x_valid = np.sort(rng.uniform(0, 1, 50)); y_valid = np.sin(2 * np.pi * x_valid) + rng.normal(0, 0.2, 50)
grid = np.linspace(0, 1, 300)
degrees = list(range(1, 16))

def fit(deg):
    coefs = np.polynomial.polynomial.polyfit(x_train, y_train, deg)
    pred = lambda v: np.polynomial.polynomial.polyval(v, coefs)
    return pred, np.mean((pred(x_train) - y_train) ** 2), np.mean((pred(x_valid) - y_valid) ** 2)

results = [fit(d) for d in degrees]
fig, (ax, ax_err) = plt.subplots(1, 2, figsize=(10, 3.8))
ax.scatter(x_train, y_train, label="training points", zorder=3)
ax.plot(grid, np.sin(2 * np.pi * grid), "--", color="grey", label="true function")
curve, = ax.plot([], [], color="crimson", lw=2, label="model")
ax.set(ylim=(-1.8, 1.8)); ax.legend(loc="lower left", fontsize=8)
ax_err.set(yscale="log", xlim=(0.5, 15.5), ylim=(1e-3, 1e2), xlabel="polynomial degree", title="Error")
train_line, = ax_err.plot([], [], "o-", label="training error")
valid_line, = ax_err.plot([], [], "o-", label="validation error")
ax_err.legend(fontsize=8); fig.tight_layout()

def update(i):
    pred, _, _ = results[i]
    curve.set_data(grid, pred(grid))
    label = "underfitting" if degrees[i] < 3 else "good fit" if degrees[i] <= 6 else "overfitting"
    ax.set_title(f"degree {degrees[i]}: {label}")
    train_line.set_data(degrees[: i + 1], [r[1] for r in results[: i + 1]])
    valid_line.set_data(degrees[: i + 1], [r[2] for r in results[: i + 1]])
    return curve, train_line, valid_line

FuncAnimation(fig, update, frames=len(degrees), interval=600).save(IMAGES / "under-overfitting.gif",
                                                                   writer=PillowWriter(fps=1.5))
plt.close(fig)
for d, (_, tr, va) in zip(degrees, results):
    if d in (1, 4, 15):
        print(f"degree {d:>2}: train MSE {tr:.3f} | validation MSE {va:.3f}")

degree  1: train MSE 0.180 | validation MSE 0.237
degree  4: train MSE 0.043 | validation MSE 0.046
degree 15: train MSE 0.010 | validation MSE 32864.446


![Underfitting to overfitting as polynomial degree grows](images/under-overfitting.gif)

<a id="the-projects-in-this-post"></a>

### The Projects in This Post

Every project later in this post follows these six steps. Here they are side by side, with the lesson each one adds.

| Project | Data | Problem | Model | Main lesson |
|---|---|---|---|---|
| Heart disease | Table, 303 patients | Binary classification | Logistic regression, KNN, random forest | Baselines, tuning, metrics beyond accuracy |
| Spam filter | E-mail text | Binary classification | Multinomial Naive Bayes | Bag of words, pipelines avoid leakage |
| Yelp reviews | Review text | Binary classification | Naive Bayes | Imbalance, per-class recall |
| Movie recommender | User × movie ratings | Ranking | Item-item correlation | Popularity bias, minimum overlap |
| Bulldozer prices | Table, 400,000 auctions | Regression over time | Random forest, gradient boosting | Date features, splitting by time |
| Chicago crime | Daily counts | Forecasting | Prophet | Resampling, data gaps, seasonality |
| Avocado prices | Weekly prices × 54 regions | Forecasting | Prophet | One series at a time, naive baselines |
| Car purchase amount | Table, 500 rows | Regression | Small dense network | Scale after splitting; a linear baseline can win |
| CIFAR-10 | 32×32 colour images | 10-class classification | CNN | Convolutions, dropout, augmentation, saving models |
| Traffic signs | 32×32 images | 43-class classification | LeNet-5 | Grayscale, normalisation, error analysis |
| Dog breeds | Photos | 120-class classification | Pretrained EfficientNetV2 | Transfer learning, tf.data pipelines |

<a id="cheatsheet-the-framework"></a>

### Cheatsheet: The Framework

| Step | Key question | Output |
|---|---|---|
| 1. Problem | Classification, regression, clustering, recommendation? | one-sentence problem statement |
| 2. Data | Structured or unstructured, static or streaming, where from? | data dictionary |
| 3. Evaluation | Which metric, and what value makes it worthwhile? | e.g. "≥ 95% accuracy", "MAE < $5k" |
| 4. Features | Which inputs could predict the target, and what's their coverage? | exploratory analysis |
| 5. Modelling | Which model, tuned how, compared how? | train/valid/test split, baseline, tuned model |
| 6. Experimentation | What else could improve results? | experiment log |

<a id="3-setting-up-a-workspace"></a>

## 3. Setting Up a Workspace

A reproducible environment means your project still runs next month, and on someone else's machine. Conda is a common
choice for data science because it manages Python and compiled libraries together.

<a id="conda-environments"></a>

### Conda Environments

- **Anaconda** is a big distribution with hundreds of data-science packages preinstalled (several GB).
- **Miniconda** contains just Python and conda, so you install only what you need. **Miniforge** is the same idea with the
  community `conda-forge` channel as the default, which also avoids Anaconda's commercial licence terms for larger
  organizations.
- **conda** itself is the package and environment manager. An **environment** is an isolated folder with its own Python
  and package versions, so upgrading pandas in one project can't break another.

Keeping the environment inside the project folder (`--prefix ./env`) makes it obvious which environment belongs to which
project. For pure-Python projects, `uv` or `python -m venv` with pip are lighter alternatives.

```bash
mkdir my-project && cd my-project
conda create --prefix ./env python=3.12 pandas numpy matplotlib scikit-learn jupyter
conda activate ./env
jupyter lab                                   # or: jupyter notebook

conda install seaborn                         # add packages later
conda env export --prefix ./env > environment.yml
conda env create --file environment.yml --prefix ./env   # recreate it elsewhere
```

<a id="working-in-jupyter"></a>

### Working in Jupyter

A notebook mixes **code cells** and **Markdown cells**, so code, results and explanation live side by side. A few habits
make notebooks much nicer to work in:

- **Shift + Enter** runs a cell. In command mode (**Esc**), **A**/**B** insert a cell above/below, **D D** deletes, and
  **M**/**Y** switch a cell to Markdown/code.
- **Shift + Tab** inside a function call shows its signature and docstring, and `func?` on its own line opens the full
  help.
- The **kernel** is the Python process behind the notebook. Cells can run in any order, so before sharing a notebook use
  *Restart Kernel and Run All* to check it works top to bottom.
- Magic commands such as `%timeit` measure how long code takes:

In [4]:
%timeit sum(range(10_000))           # pure Python loop
%timeit np.arange(10_000).sum()      # vectorized NumPy - typically far faster

145 μs ± 8.39 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


6.14 μs ± 300 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
